[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/trigonometry_course.ipynb)

# Trigonometry full course: practice notebook

One practice question for every teaching chapter of the video, in the same order. Work each one out on paper first,
then type your answer into the box and press **check**. It tells you whether you're right. Press **show working**
if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. Change any number and run the cell again: nothing can
break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Fractions as `20/29`, roots as `sqrt(3)/2`, π as `pi`, degrees as plain numbers (`150`). For
several answers, separate them with commas, in the order the question asks.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # cosine · sine · tangent

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x = sp.Symbol("x")
r = sp.Rational
deg = lambda v: v * sp.pi / 180

def read(s):
    """Your typed answer as an exact number (or a list of them)."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("π", "pi").replace("°", "")
          .replace("cosec", "csc"))
    s = re.sub(r"√\s*(\d+(?:\.\d+)?|pi|\w)", r"sqrt(\1)", s).replace("√", "sqrt")   # √3 → sqrt(3); √(…) stays
    return [parse_expr(p, local_dict={"pi": sp.pi, "e": sp.E}, transformations=RULES) for p in s.split(",") if p.strip()]

def same(a, b):
    try: return abs(complex(sp.N(a - b, 30))) < 1e-9
    except TypeError: return False                         # a letter that isn't pi: not a number

# Each question: the question, the exact answers in order, whether order matters, and the working.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="420px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            if q.get("any_order"):
                left = list(want); ok = all(any(same(g, w) and not left.remove(w) for w in list(left)) for g in got)
            else:
                ok = all(same(g, w) for g, w in zip(got, want))
            print("✓ right" if ok else "✗ not quite: try again, or press show working")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def check_identity(left, right, start=-360, stop=360, step=1):
    """Checks that two sides agree at every angle from start to stop (degrees), skipping angles where either side is
    undefined. Prints a few rows of the table, and the first angle where they part company, if there is one."""
    theta = sp.Symbol("theta")
    fix = lambda s: s.replace("θ", "theta").replace("−", "-").replace("cosec", "csc").replace("²", "^2")
    L = parse_expr(fix(left), local_dict={"theta": theta}, transformations=RULES)
    R = parse_expr(fix(right), local_dict={"theta": theta}, transformations=RULES)
    fL, fR = sp.lambdify(theta, L, "math"), sp.lambdify(theta, R, "math")
    tried, shown = 0, 0
    print(f"{'angle':>7}   {'left side':>12}   {'right side':>12}")
    for a in range(start, stop + 1, step):
        try: vl, vr = fL(math.radians(a)), fR(math.radians(a))
        except (ZeroDivisionError, ValueError, OverflowError): continue
        if abs(vl) > 1e6 or abs(vr) > 1e6: continue        # next to an angle where a side is undefined
        tried += 1
        if abs(vl - vr) > 1e-9 * max(1, abs(vl)):
            print(f"{a:>6}°   {vl:>12.6f}   {vr:>12.6f}")
            print(f"✗ the two sides part company at {a}°: they are not the same function."); return
        if a % 60 == 0 and shown < 6: print(f"{a:>6}°   {vl:>12.6f}   {vr:>12.6f}"); shown += 1
    print(f"✓ equal at all {tried} angles tried, from {start}° to {stop}°. Now write the proof line by line.")

def identity_checker(left, right):
    """Two boxes and a check button for check_identity."""
    lb = Text(value=left, description="left:", layout=Layout(width="380px"))
    rb = Text(value=right, description="right:", layout=Layout(width="380px"))
    b, out = Button(description="check"), Output()
    def on(_=None):
        out.clear_output()
        with out:
            try: check_identity(lb.value, rb.value)
            except Exception: print("I can't read that yet: write θ as theta and use ^ for powers.")
    b.on_click(on); display(VBox([HBox([lb, rb, b]), out])); on()

## 1 · Trigonometric ratios

In a right triangle, pick an angle θ. **sin θ** = opposite ÷ hypotenuse, **cos θ** = adjacent ÷ hypotenuse,
**tan θ** = opposite ÷ adjacent. Their upside-down partners: cosec θ = 1/sin θ, sec θ = 1/cos θ, cot θ = 1/tan θ.

In [ ]:
Q[1] = {"text": "A right triangle has sides 20, 21 and 29. The angle θ is across from the side 20. Give sin θ, cos θ, "
                "tan θ, cosec θ, sec θ and cot θ, in that order.",
        "answer": [r(20, 29), r(21, 29), r(20, 21), r(29, 20), r(29, 21), r(21, 20)],
        "working": ["Check it's a right triangle: 20² + 21² = 400 + 441 = 841 = 29².",
                    "Hypotenuse 29 (the longest side). Opposite θ: 20. Adjacent: 21.",
                    "sin θ = 20/29, cos θ = 21/29, tan θ = 20/21.",
                    "Flip each: cosec θ = 29/20, sec θ = 29/21, cot θ = 21/20."]}
assert 20 ** 2 + 21 ** 2 == 29 ** 2
practice(1)

## 2 · Special angles

From the half square: sin 45° = cos 45° = 1/√2, tan 45° = 1. From half an equilateral triangle: sin 30° = cos 60° = ½,
sin 60° = cos 30° = √3/2, tan 30° = 1/√3, tan 60° = √3.

In [ ]:
Q[2] = {"text": "Without a calculator, find the exact value of 2 sin²60° − tan²45° + 4 cos²30°.",
        "answer": [r(7, 2)],
        "working": ["sin 60° = √3/2, so sin²60° = 3/4, and 2 × 3/4 = 3/2.",
                    "tan 45° = 1, so tan²45° = 1.",
                    "cos 30° = √3/2, so cos²30° = 3/4, and 4 × 3/4 = 3.",
                    "3/2 − 1 + 3 = 7/2."]}
assert sp.simplify(2 * sp.sin(deg(60)) ** 2 - sp.tan(deg(45)) ** 2 + 4 * sp.cos(deg(30)) ** 2 - r(7, 2)) == 0
practice(2)

## 3 · Angles: degrees and radians

π radians = 180°. Arc length l = rθ and sector area ½r²θ, both with θ **in radians**.

In [ ]:
Q[3] = {"text": "A sector of a circle has radius 15 cm and angle 72°. Find its arc length, exactly, in cm. (Change 72° "
                "to radians first.)",
        "answer": [6 * sp.pi],
        "working": ["72° × π/180 = 2π/5 radians.",
                    "l = rθ = 15 × 2π/5 = 6π cm, about 18.85 cm.",
                    "Using 72 instead of 2π/5 would give 1,080 cm: the formula needs radians."]}
assert sp.simplify(15 * deg(72) - 6 * sp.pi) == 0
practice(3)

## 4 · The unit circle

Turn a point on a circle of radius 1 through θ: it lands at (cos θ, sin θ). The quadrant gives the signs, and the
reference angle gives the size. Drag θ and watch the two parts change sign.

In [ ]:
def unit_circle(theta=150):
    t = math.radians(theta)
    px, py = math.cos(t), math.sin(t)
    a = np.linspace(0, 2 * np.pi, 300)
    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    ax.plot(np.cos(a), np.sin(a), color=DIM)
    ax.plot([0, px], [0, py], color=CORAL, lw=3)
    ax.plot([0, px], [0, 0], color=TEAL, lw=5, label=f"cos {theta}° = {px:+.3f}")
    ax.plot([px, px], [0, py], color=SAGE, lw=5, label=f"sin {theta}° = {py:+.3f}")
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect("equal"); ax.legend(loc="lower left")
    plt.show()

interact(unit_circle, theta=FloatSlider(value=150, min=-360, max=360, step=5, description="θ (°)"));

In [ ]:
Q[4] = {"text": "sin θ = −5/13, and θ is in the fourth quadrant. Give cos θ and tan θ, in that order.",
        "answer": [r(12, 13), r(-5, 12)],
        "working": ["A triangle with opposite 5 and hypotenuse 13 has adjacent √(169 − 25) = 12.",
                    "In the fourth quadrant the across part (cosine) is positive and the height (sine) negative.",
                    "cos θ = 12/13, tan θ = sin θ ÷ cos θ = −5/12."]}
assert r(12, 13) ** 2 + r(-5, 13) ** 2 == 1
practice(4)

## 5 · Graphs

y = a sin(bx) + c: the amplitude is |a|, the period is 2π/b, and c lifts the whole wave. Drag the sliders.

In [ ]:
def wave(a=3.0, b=2.0, c=1.0):
    xs = np.linspace(-0.2, 2 * np.pi + 0.2, 500)
    fig, ax = plt.subplots(figsize=(8.4, 4.2))
    ax.plot(xs, np.sin(xs), color=DIM, lw=1.4, label="y = sin x")
    ax.plot(xs, a * np.sin(b * xs) + c, color=SAGE, label=f"y = {a:g} sin({b:g}x) + {c:g}")
    ax.axhline(c, color=CORAL, lw=1, ls="--")
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi], ["0", "π/2", "π", "3π/2", "2π"])
    ax.set_ylim(-6, 6); ax.legend(loc="lower left")
    ax.set_title(f"amplitude {abs(a):g}   ·   period 2π/{b:g} = {2 * math.pi / b:.3f}")
    plt.show()

interact(wave, a=FloatSlider(value=3, min=-4, max=4, step=0.5), b=FloatSlider(value=2, min=0.5, max=4, step=0.5),
         c=FloatSlider(value=1, min=-2, max=2, step=0.5));

In [ ]:
Q[5] = {"text": "For y = 3 sin(2x) + 1, give the amplitude, the period (in radians) and the greatest value of y, in "
                "that order.",
        "answer": [3, sp.pi, 4],
        "working": ["Amplitude: the number in front of the sine, 3.",
                    "Period: 2π ÷ 2 = π, because the wave runs twice as fast.",
                    "Greatest value: the sine's top is 1, so 3 × 1 + 1 = 4."]}
practice(5)

## 6 · Identities

An identity is true at **every** angle. Prove it by working on one side until it becomes the other. Before you write
the proof, the checker below tests your two sides at hundreds of angles at once. If they ever disagree, it shows you
the first angle where they part company. Write θ as `theta`, powers with `^`.

**Practice:** prove that sec θ − cos θ = sin θ tan θ.

In [ ]:
identity_checker("sec(theta) - cos(theta)", "sin(theta)*tan(theta)")

A tempting one that isn't an identity: try `sin(2*theta)` against `2*sin(theta)`, and see where they part company.

Working, if you want it: sec θ − cos θ = 1/cos θ − cos θ = (1 − cos²θ)/cos θ = sin²θ/cos θ = sin θ × (sin θ/cos θ)
= sin θ tan θ.

## 7 · Compound angles

sin(A ± B) = sin A cos B ± cos A sin B, and cos(A ± B) = cos A cos B ∓ sin A sin B.

In [ ]:
Q[7] = {"text": "Find the exact value of sin 15°, using 15° = 45° − 30°.",
        "answer": [(sp.sqrt(6) - sp.sqrt(2)) / 4],
        "working": ["sin(45° − 30°) = sin 45° cos 30° − cos 45° sin 30°",
                    "= (1/√2)(√3/2) − (1/√2)(1/2)",
                    "= (√3 − 1)/(2√2) = (√6 − √2)/4, about 0.2588."]}
assert sp.simplify(sp.sin(deg(15)) - (sp.sqrt(6) - sp.sqrt(2)) / 4) == 0
practice(7)

## 8 · Double, half and triple angles

sin 2θ = 2 sin θ cos θ; cos 2θ = cos²θ − sin²θ = 1 − 2 sin²θ = 2 cos²θ − 1.

In [ ]:
Q[8] = {"text": "cos θ = 3/5 and θ is acute. Find sin 2θ and cos 2θ, in that order.",
        "answer": [r(24, 25), r(-7, 25)],
        "working": ["sin θ = 4/5 (the 3-4-5 triangle).",
                    "sin 2θ = 2 × 4/5 × 3/5 = 24/25.",
                    "cos 2θ = 2 cos²θ − 1 = 18/25 − 1 = −7/25."]}
assert 2 * r(4, 5) * r(3, 5) == r(24, 25) and 2 * r(3, 5) ** 2 - 1 == r(-7, 25)
practice(8)

## 9 · Sums and products

cos C + cos D = 2 cos((C + D)/2) cos((C − D)/2), and the three others like it.

In [ ]:
Q[9] = {"text": "Find the exact value of cos 75° + cos 15°.",
        "answer": [sp.sqrt(6) / 2],
        "working": ["cos 75° + cos 15° = 2 cos 45° cos 30°",
                    "= 2 × (1/√2) × (√3/2) = √3/√2 = √6/2, about 1.2247."]}
assert sp.simplify(sp.cos(deg(75)) + sp.cos(deg(15)) - sp.sqrt(6) / 2) == 0
practice(9)

## 10 · Trigonometric equations

Find the reference angle, use the sign to pick the quadrants, then write every answer in the interval.

In [ ]:
Q[10] = {"text": "Solve 2 cos x + √3 = 0 for 0° ≤ x < 360°. Give every solution in degrees, separated by commas.",
         "answer": [150, 210], "any_order": True,
         "working": ["cos x = −√3/2. The reference angle is 30°, because cos 30° = √3/2.",
                     "Cosine is negative in the second and third quadrants.",
                     "x = 180° − 30° = 150°, and x = 180° + 30° = 210°."]}
assert all(abs(2 * math.cos(math.radians(a)) + math.sqrt(3)) < 1e-12 for a in (150, 210))
practice(10)

## 11 · Inverse trigonometric functions

tan⁻¹x is the angle between −90° and 90° whose tangent is x. It is not 1/tan x.

In [ ]:
Q[11] = {"text": "Find the exact value of cos(tan⁻¹(3/4)).",
         "answer": [r(4, 5)],
         "working": ["Let θ = tan⁻¹(3/4): an angle between −90° and 90° with tan θ = 3/4. It's positive, so acute.",
                     "Draw it: opposite 3, adjacent 4, hypotenuse 5.",
                     "cos θ = 4/5."]}
assert sp.simplify(sp.cos(sp.atan(r(3, 4))) - r(4, 5)) == 0
practice(11)

## 12 · The sine rule and the cosine rule

a/sin A = b/sin B = c/sin C, and a² = b² + c² − 2bc cos A.

In [ ]:
Q[12] = {"text": "Two sides of a triangle are 5 cm and 7 cm, with 60° between them. Find the third side, exactly, in cm.",
         "answer": [sp.sqrt(39)],
         "working": ["The angle between the two known sides: use the cosine rule.",
                     "c² = 5² + 7² − 2 × 5 × 7 × cos 60° = 25 + 49 − 35 = 39.",
                     "c = √39, about 6.24 cm."]}
assert 25 + 49 - 2 * 5 * 7 * r(1, 2) == 39
practice(12)

## 13 · Heights and distances

The angle of elevation is measured up from the level. Height = distance × tan(angle).

In [ ]:
Q[13] = {"text": "You stand 40 m from the foot of a tree. The angle of elevation of its top is 30°. How tall is the "
                 "tree, exactly, in metres?",
         "answer": [40 * sp.sqrt(3) / 3],
         "working": ["tan 30° = height ÷ 40.",
                     "height = 40 × tan 30° = 40/√3 = 40√3/3, about 23.09 m."]}
assert sp.simplify(40 * sp.tan(deg(30)) - 40 * sp.sqrt(3) / 3) == 0
practice(13)

## Sources
NCERT Mathematics Class X, ch. 8–9; Class XI, ch. 3 (Trigonometric Functions); Class XII, ch. 2 (Inverse
Trigonometric Functions); OpenStax, Algebra and Trigonometry 2e, ch. 7–10. All retrieved 2026-10-29. Every question
is our own.